# ProteinOPD Training Demo

This notebook runs a small Colab-friendly demo of the ProteinOPD training flow.

The workflow is split into two explicit stages:

1. Train one teacher adapter.
2. Train an OPD student adapter using 1-3 teacher adapters.

Teacher adapters are local paths produced in this notebook or supplied by the user. This notebook does not select teacher adapters from Hugging Face and does not publish adapters to Hugging Face.

Use a GPU runtime: `Runtime > Change runtime type > T4/L4/A100 GPU`.

In [ ]:
#@title Setup
repo_url = "https://github.com/THU-AI4S/ProteinOPD.git" #@param {type:"string"}
branch = "main" #@param {type:"string"}
mount_google_drive = True #@param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    print("$", " ".join(str(x) for x in cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    result.check_returncode()


run(["nvidia-smi"])

repo_dir = Path("/content/ProteinOPD")
if repo_dir.exists():
    run(["git", "fetch", "origin"], cwd=repo_dir)
    run(["git", "checkout", branch], cwd=repo_dir)
    run(["git", "pull", "--ff-only"], cwd=repo_dir)
else:
    run(["git", "clone", "--branch", branch, repo_url, str(repo_dir)])

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])

PINNED_PACKAGES = [
    "accelerate==1.12.0",
    "datasets==4.4.2",
    "huggingface_hub",
    "peft==0.18.0",
    "pyyaml",
    "safetensors==0.7.0",
    "sentencepiece==0.2.1",
    "tensorboard",
    "tokenizers==0.22.2",
    "tqdm",
    "transformers==4.57.3",
    "wandb",
]
run([sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES])

import importlib.metadata as md
for pkg in ["torch", "transformers", "peft", "accelerate", "datasets", "huggingface_hub", "tokenizers", "safetensors", "sentencepiece", "torchao"]:
    try:
        print(f"{pkg}=={md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg} not installed")

if mount_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")

print("Repository:", repo_dir)

In [ ]:
#@title Optional Hugging Face login for gated base models
login_to_huggingface = False #@param {type:"boolean"}

if login_to_huggingface:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Skipping Hugging Face login. Enable this only if the base model is private or gated.")

In [ ]:
import torch
from huggingface_hub import snapshot_download

if not torch.cuda.is_available():
    raise RuntimeError("This demo requires a GPU runtime.")

precision_flag = "--bf16" if torch.cuda.is_bf16_supported() else "--fp16"
print("Using precision:", precision_flag)

work_dir = Path("/content/proteinopd_training_demo")
work_dir.mkdir(parents=True, exist_ok=True)


def resolve_local_model(model_id_or_path):
    candidate = Path(model_id_or_path).expanduser()
    if candidate.exists():
        return str(candidate.resolve())
    print("Downloading model snapshot from Hugging Face:", model_id_or_path)
    return snapshot_download(repo_id=model_id_or_path)


def dataset_path(track, preference_dataset, split):
    name = "train" if split == "train" else "test"
    suffix = "200" if split == "train" else "50"
    if track == "unconditional":
        if preference_dataset == "sol":
            filename = f"{name}_sol_{suffix}.csv"
        elif preference_dataset == "foldability":
            filename = f"{name}_plddt_{suffix}.csv"
        else:
            filename = f"{name}_thermo_{suffix}.csv"
        return repo_dir / "unconditional" / "data" / preference_dataset / filename
    if preference_dataset == "sol":
        filename = f"{name}_sol_{suffix}.json"
    elif preference_dataset == "foldability":
        filename = f"{name}_plddt_{suffix}.json"
    else:
        filename = f"{name}_thermo_{suffix}.json"
    return repo_dir / "conditional" / "data" / preference_dataset / filename

from transformers import TrainingArguments
import inspect
eval_strategy_arg = "--eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "--evaluation_strategy"
print("Using evaluation strategy argument:", eval_strategy_arg)


## Stage 1: Train a Teacher Adapter

Run this stage once per preference dataset. After each run, copy the printed `teacher_adapter_path` into one of the OPD teacher path fields in Stage 2.

In [ ]:
#@title Teacher training settings
teacher_track = "unconditional" #@param ["unconditional", "conditional"]
teacher_preference_dataset = "sol" #@param ["sol", "foldability", "thermo"]

protgpt2_model_id = "nferruz/ProtGPT2" #@param {type:"string"}
prollama_model_id = "GreatCaptainNemo/ProLLaMA" #@param {type:"string"}

teacher_output_root = "/content/drive/MyDrive/ProteinOPD/teachers" #@param {type:"string"}
teacher_epochs = 1 #@param {type:"integer"}
teacher_batch_size = 1 #@param {type:"integer"}
teacher_gradient_accumulation_steps = 1 #@param {type:"integer"}
teacher_max_length = 256 #@param {type:"integer"}

teacher_output_root = Path(teacher_output_root)
teacher_output_root.mkdir(parents=True, exist_ok=True)

teacher_train_data = dataset_path(teacher_track, teacher_preference_dataset, "train")
teacher_eval_data = dataset_path(teacher_track, teacher_preference_dataset, "test")
print("Teacher train data:", teacher_train_data)
print("Teacher eval data:", teacher_eval_data)
print("Teacher output root:", teacher_output_root)

In [ ]:
#@title Run teacher training
import gc
import os
import yaml

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


def train_unconditional_teacher():
    local_protgpt2_model_path = resolve_local_model(protgpt2_model_id)
    teacher_output = teacher_output_root / f"unconditional_{teacher_preference_dataset}_prefix"
    teacher_config_path = work_dir / "unconditional_teacher.yaml"
    teacher_config = {
        "model_name_or_path": local_protgpt2_model_path,
        "tokenizer_name_or_path": local_protgpt2_model_path,
        "train_path": str(teacher_train_data),
        "test_path": str(teacher_eval_data),
        "text_column": "sequence",
        "dataset_name": f"proteinopd_{teacher_preference_dataset}_demo",
        "output_dir": str(teacher_output),
        "log_dir": str(work_dir / "logs" / "unconditional_teacher"),
        "run_dir": str(work_dir / "runs" / "unconditional_teacher"),
        "num_virtual_tokens": 8,
        "max_length": teacher_max_length,
        "learning_rate": 0.005,
        "num_epochs": teacher_epochs,
        "batch_size": teacher_batch_size,
        "seed": 42,
        "device": "auto",
        "early_stop": False,
    }
    teacher_config_path.write_text(yaml.safe_dump(teacher_config), encoding="utf-8")
    run([sys.executable, "unconditional/teacher_construct/prefix_tuning_prot.py", "--config", str(teacher_config_path)], cwd=repo_dir)
    adapter_dirs = sorted(teacher_output.glob("prefix_*"))
    if not adapter_dirs:
        raise RuntimeError(f"No teacher adapter found under {teacher_output}")
    return adapter_dirs[-1]


def train_conditional_teacher():
    local_prollama_model_path = resolve_local_model(prollama_model_id)
    teacher_output = teacher_output_root / f"conditional_{teacher_preference_dataset}_lora"
    teacher_script = repo_dir / "conditional" / "teacher_construct" / "scripts" / "instruction_tune.py"
    cmd = [
        sys.executable, str(teacher_script),
        "--model_name_or_path", local_prollama_model_path,
        "--tokenizer_name_or_path", local_prollama_model_path,
        "--train_file", str(teacher_train_data),
        "--validation_file", str(teacher_eval_data),
        "--per_device_train_batch_size", str(teacher_batch_size),
        "--per_device_eval_batch_size", str(teacher_batch_size),
        "--do_train", "--do_eval", "--seed", "42",
        precision_flag,
        "--num_train_epochs", str(teacher_epochs),
        "--lr_scheduler_type", "cosine",
        "--learning_rate", "2e-5",
        "--warmup_ratio", "0.05",
        "--logging_strategy", "steps",
        "--logging_steps", "1",
        eval_strategy_arg, "steps",
        "--eval_steps", "20",
        "--save_strategy", "steps",
        "--save_steps", "50",
        "--save_total_limit", "2",
        "--gradient_accumulation_steps", str(teacher_gradient_accumulation_steps),
        "--preprocessing_num_workers", "1",
        "--max_seq_length", str(teacher_max_length),
        "--output_dir", str(teacher_output),
        "--overwrite_output_dir",
        "--report_to", "none",
        "--lora_rank", "8",
        "--lora_alpha", "16",
        "--trainable", "q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj",
        "--lora_dropout", "0.1",
        "--torch_dtype", "bfloat16" if precision_flag == "--bf16" else "float16",
        "--load_in_kbits", "16",
        "--save_safetensors", "False",
        "--gradient_checkpointing",
    ]
    run(cmd, cwd=teacher_script.parent)
    return teacher_output


teacher_adapter_path = train_unconditional_teacher() if teacher_track == "unconditional" else train_conditional_teacher()
print("teacher_adapter_path =", teacher_adapter_path)

## Stage 2: Train the OPD Student Adapter

Choose 1-3 local teacher adapters. For conditional OPD, all teachers must use the same adapter type, and the ProLLaMA backbone is downloaded once to a local Colab path before training.

In [ ]:
#@title OPD training settings
opd_track = "unconditional" #@param ["unconditional", "conditional"]

opd_teacher_1_path = "" #@param {type:"string"}
opd_teacher_1_weight = 1.0 #@param {type:"number"}
opd_teacher_2_path = "" #@param {type:"string"}
opd_teacher_2_weight = 1.0 #@param {type:"number"}
opd_teacher_3_path = "" #@param {type:"string"}
opd_teacher_3_weight = 1.0 #@param {type:"number"}

opd_protgpt2_model_id = "nferruz/ProtGPT2" #@param {type:"string"}
opd_prollama_model_id = "GreatCaptainNemo/ProLLaMA" #@param {type:"string"}

opd_output_root = "/content/drive/MyDrive/ProteinOPD/opd_students" #@param {type:"string"}
num_train_samples = 64 #@param {type:"integer"}
student_batch_size = 1 #@param {type:"integer"}
student_gradient_accumulation_steps = 1 #@param {type:"integer"}
max_new_tokens = 128 #@param {type:"integer"}

opd_output_root = Path(opd_output_root)
opd_output_root.mkdir(parents=True, exist_ok=True)

teacher_paths_and_weights = [
    (opd_teacher_1_path.strip(), float(opd_teacher_1_weight)),
    (opd_teacher_2_path.strip(), float(opd_teacher_2_weight)),
    (opd_teacher_3_path.strip(), float(opd_teacher_3_weight)),
]
opd_teachers = [(Path(path).expanduser(), weight) for path, weight in teacher_paths_and_weights if path]
if not 1 <= len(opd_teachers) <= 3:
    raise ValueError("Provide 1-3 local teacher adapter paths for OPD training.")
for teacher_path, weight in opd_teachers:
    if not teacher_path.exists():
        raise FileNotFoundError(f"Teacher adapter path does not exist: {teacher_path}")
    if weight < 0:
        raise ValueError(f"Teacher weight must be non-negative: {teacher_path}")
if sum(weight for _, weight in opd_teachers) <= 0:
    raise ValueError("At least one teacher weight must be > 0.")

print("OPD teachers:")
for teacher_path, weight in opd_teachers:
    print("-", teacher_path, "weight=", weight)
print("OPD output root:", opd_output_root)

In [ ]:
#@title Run OPD student training
import yaml


def train_unconditional_opd():
    local_protgpt2_model_path = resolve_local_model(opd_protgpt2_model_id)
    teacher_config_path = work_dir / "unconditional_opd_teachers.yaml"
    unconditional_teachers = list(opd_teachers)
    if len(unconditional_teachers) == 1:
        path, weight = unconditional_teachers[0]
        unconditional_teachers = [(path, weight / 2.0), (path, weight / 2.0)]
        print("Unconditional OPD requires at least two teacher entries; duplicating the single selected teacher with split weights.")
    teacher_config = {
        "log_teacher_entropy": False,
        "teacher_backbone_path": local_protgpt2_model_path,
        "teachers": [
            {
                "name": f"teacher_{idx}",
                "adapter_path": str(path),
                "weight": weight,
                "temperature": 1.0,
            }
            for idx, (path, weight) in enumerate(unconditional_teachers, start=1)
        ],
    }
    teacher_config_path.write_text(yaml.safe_dump(teacher_config), encoding="utf-8")
    student_output = opd_output_root / "unconditional_student_opd"
    cmd = [
        sys.executable, "unconditional/proteinopd/protein_opd_train.py",
        "--num_train_samples", str(num_train_samples),
        "--student_model_name_or_path", local_protgpt2_model_path,
        "--teacher_config_path", str(teacher_config_path),
        "--prompt_mode", "unconditional",
        "--max_new_tokens", str(max_new_tokens),
        "--temperature", "1.0",
        "--repetition_penalty", "1.2",
        "--top_p", "0.95",
        "--top_k", "500",
        "--beta", "0.5",
        "--useloss", "jsd",
        "--student_tune_mode", "lora",
        "--lora_r", "8",
        "--lora_alpha", "16",
        "--lora_dropout", "0.05",
        "--lora_target_modules", "c_attn", "c_proj", "c_fc",
        "--per_device_train_batch_size", str(student_batch_size),
        "--gradient_accumulation_steps", str(student_gradient_accumulation_steps),
        "--learning_rate", "2e-5",
        "--num_train_epochs", "1.0",
        "--logging_steps", "1",
        "--save_steps", "50",
        "--report_to", "none",
        "--output_dir", str(student_output),
        "--overwrite_output_dir",
        precision_flag,
    ]
    run(cmd, cwd=repo_dir)
    return student_output


def train_conditional_opd():
    local_prollama_model_path = resolve_local_model(opd_prollama_model_id)
    opd_config_path = work_dir / "conditional_opd.yaml"
    opd_config = {
        "student": {
            "backbone_path": local_prollama_model_path,
            "lora": {
                "r": 8,
                "alpha": 16,
                "dropout": 0.1,
                "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
            },
        },
        "teachers": {
            "backbone_path": local_prollama_model_path,
            "adapters": [
                {
                    "name": f"teacher_{idx}",
                    "adapter_path": str(path),
                    "weight": weight,
                    "temperature": 0.7,
                }
                for idx, (path, weight) in enumerate(opd_teachers, start=1)
            ],
        },
        "prompt": {
            "instruction": "[Generate by superfamily]",
            "input": "Superfamily=<Lysozyme-like domain superfamily>",
            "sequence_regex": "Seq=<([^>]*)>",
        },
        "distill": {
            "loss_type": "jsd",
            "beta": 0.5,
            "top_k_loss": 0,
            "use_z_t": False,
            "use_entropy_rule": False,
            "entropy_alpha": 0.0,
            "log_teacher_entropy": False,
            "use_wi_entropy": False,
            "use_wi_distance": False,
            "wi_entropy_tau": 1.0,
            "wi_distance_beta": 0.9,
            "wi_distance_tau": 1.0,
            "protein_opd": {"debug_teacher_diff": False, "debug_teacher_diff_steps": 5},
        },
        "generation": {
            "num_train_samples": num_train_samples,
            "max_new_tokens": max_new_tokens,
            "student_temperature": 1.0,
            "use_curriculum_tem": False,
            "curriculum_tem_start": 0.7,
            "repetition_penalty": 1.2,
            "top_p": 0.9,
            "top_k": 200,
            "save_generations": True,
            "generation_save_steps": 50,
        },
    }
    opd_config_path.write_text(yaml.safe_dump(opd_config), encoding="utf-8")
    student_output = opd_output_root / "conditional_student_opd"
    cmd = [
        sys.executable, "conditional/proteinopd/prollama_opd_train.py",
        "--opd_config_path", str(opd_config_path),
        "--output_dir", str(student_output),
        "--overwrite_output_dir",
        "--per_device_train_batch_size", str(student_batch_size),
        "--gradient_accumulation_steps", str(student_gradient_accumulation_steps),
        "--do_train", "--seed", "42",
        precision_flag,
        "--num_train_epochs", "1.0",
        "--lr_scheduler_type", "cosine",
        "--learning_rate", "2e-5",
        "--warmup_ratio", "0.1",
        "--logging_strategy", "steps",
        "--logging_steps", "1",
        "--report_to", "none",
        "--save_strategy", "steps",
        "--save_steps", "50",
        "--save_total_limit", "2",
        eval_strategy_arg, "no",
        "--load_in_kbits", "16",
        "--save_safetensors", "False",
        "--gradient_checkpointing",
    ]
    run(cmd, cwd=repo_dir)
    return student_output


student_output = train_unconditional_opd() if opd_track == "unconditional" else train_conditional_opd()
print("student_output =", student_output)